ST554 Project 2

Franklin Zhou

Mar 9 2026

# Part I


## Introduction

I'm going to use the `SparkDataCheck.py` to do the following tasks:

- validation checks

- summarization functions

- chainable operations

## Initiating 

First, import the script, required modules and initiate the spark session:

In [34]:
from SparkDataCheck import SparkDataCheck
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Project2") \
    .config("spark.sql.ansi.enabled", "false") \
    .getOrCreate()

## Prepare for data

Read in the air quality data and make some data cleaning and transformation. Since the table is too wide, I select some variables for simplicity and export to a `.csv` file as an example.

In [2]:
air_data = pd.read_csv("air.csv")

# change all "-200" to missing
air_data.replace(-200, np.nan, inplace = True)

# create some categorical variables
air_data["Time_TS"] = air_data.Time.astype('timedelta64[ns]')
air_data["Time_of_Day"] = pd.cut(air_data.Time_TS, 3, labels = ["Early", "Mid", "Late"])
air_data["Temp_Range"] = pd.cut(air_data['T'], bins = [-20,0,15,30,50], labels = ["Cold", "Cool", "Warm", "Hot"])

# Pick a subset of the data as an illustration and export to .csv file
air_data.loc[:, ["CO(GT)", "NO2(GT)", "C6H6(GT)", "Time_of_Day", "Temp_Range"]].to_csv('air_data_cleaned.csv', index = False)

In [ ]:
Read the `.csv` file by using `.read_csv()` method

In [3]:
air = SparkDataCheck.read_csv(spark, "air_data_cleaned.csv")

## Examine Validation methods

Then I'll check all methods that I created in the class.

### 1. Validate numeric bounds

In [4]:
# Is the value of C6H6(GT) is between 5 and 9
air.check_numeric("NO2(GT)", lower = 50, upper = 100)
air.df.show()

+------+-------+--------+-----------+----------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|
+------+-------+--------+-----------+----------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                false|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|
|   2.2|  114.0|     9.0|       Late|      Cool|                false|
|   2.2|  122.0|     9.2|       Late|      Cool|                false|
|   1.6|  116.0|     6.5|       Late|      Cool|                false|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|
|   1.0|   76.0|     3.3|      Early|      Cool|                 true|
|   0.9|   60.0|     2.3|      Early|      Cool|                 true|
|   0.6|   NULL|     1.7|      Early|      Cool|                 NULL|
|  NULL|   34.0|     1.3|      Early|      Cool|                false|
|   0.

In [32]:
# Is the value of C6H6(GT) is above 50
air.check_numeric("NO2(GT)", lower = 50)
air.df.show()

+------+-------+--------+-----------+----------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|
+------+-------+--------+-----------+----------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|
|   1.0|   76.0|     3.3|      Early|      Cool|                 true|
|   0.9|   60.0|     2.3|      Early|      Cool|                 true|
|   0.6|   NULL|     1.7|      Early|      Cool|                 NULL|
|  NULL|   34.0|     1.3|      Early|      Cool|                false|
|   0.

In [33]:
# If the input column is not numeric
air.check_numeric("Temp_Range", lower = 50)

### 2. Validate string type

In [34]:
air.check_string("Time_of_Day", levels = ["Mid","Late"])
air.df.show()

+------+-------+--------+-----------+----------+---------------------+------------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|Time_of_Day_string_check|
+------+-------+--------+-----------+----------+---------------------+------------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|                    true|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|                    true|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|                    true|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|                    true|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|                    true|
|   1.2|   96.0|     4.7|       Late|      Cool|                 true|                    true|
|   1.2|   77.0|     3.6|      Early|      Cool|                 true|                   false|
|   1.0|   76.0|     3.3|      Early|   

In [35]:
air.check_string("NO2(GT)")

### 3. Validate missing values

In [36]:
air.check_missing("NO2(GT)")
air.df.show()

+------+-------+--------+-----------+----------+---------------------+------------------------+---------------------+
|CO(GT)|NO2(GT)|C6H6(GT)|Time_of_Day|Temp_Range|NO2(GT)_numeric_check|Time_of_Day_string_check|NO2(GT)_missing_check|
+------+-------+--------+-----------+----------+---------------------+------------------------+---------------------+
|   2.6|  113.0|    11.9|       Late|      Cool|                 true|                    true|                false|
|   2.0|   92.0|     9.4|       Late|      Cool|                 true|                    true|                false|
|   2.2|  114.0|     9.0|       Late|      Cool|                 true|                    true|                false|
|   2.2|  122.0|     9.2|       Late|      Cool|                 true|                    true|                false|
|   1.6|  116.0|     6.5|       Late|      Cool|                 true|                    true|                false|
|   1.2|   96.0|     4.7|       Late|      Cool|        

## Examine Summarization methods

### 1. find min and max of column(s)

In [5]:
# find min and max of NO2(GT)
air.min_max("NO2(GT)")

,min(NO2(GT)),max(NO2(GT))
0,2.0,340.0


In [6]:
# find min and max of NO2(GT) grouped by temp range
air.min_max("NO2(GT)", group = "Temp_Range")

,Temp_Range,min(NO2(GT)),max(NO2(GT))
0,Cool,13.0,333.0
1,Cold,27.0,169.0
2,None,35.0,340.0
3,Hot,14.0,233.0
4,Warm,2.0,272.0


In [8]:
# find min and max of all numeric columns gruped by Time of Day
air.min_max(group = "Time_of_Day")

,Time_of_Day,min(CO(GT)),max(CO(GT)),min(NO2(GT)),max(NO2(GT)),min(C6H6(GT)),max(C6H6(GT))
0,Late,0.1,11.9,21.0,340.0,1.3,52.1
1,Early,0.1,5.9,2.0,250.0,0.1,31.5
2,Mid,0.1,8.1,5.0,333.0,0.3,63.7


In [9]:
# if column is not numeric type, display warning message
air.min_max("Time_of_Day")

### 2. count by column levels

In [10]:
# by one column
air.levels_count("Time_of_Day")

,Time_of_Day,count
0,Late,3118
1,Early,3120
2,Mid,3119


In [11]:
# by two columns
air.levels_count("Time_of_Day", "Temp_Range")

,Time_of_Day,Temp_Range,count
0,Early,Warm,1465
1,Mid,Warm,1493
2,Mid,Hot,537
3,Early,Cold,11
4,Late,Cool,1018
5,Mid,Cool,976
6,Late,None,139
7,Early,None,117
8,Early,Cool,1527
9,Late,Hot,401


In [12]:
# if any column is numeric
air.levels_count("Time_of_Day", "NO2(GT)")

## Using pandas data frame

In [26]:
# Pick a subset of the data as an illustration and export to .csv file
air_data_pd = air_data.loc[:, ["CO(GT)", "NO2(GT)", "C6H6(GT)", "Time_of_Day", "Temp_Range"]]
isinstance(air_data_pd, pd.DataFrame) # varified that this is a pandas dataframe

True

In [16]:
# create an instance from pandas dataframe
air_pd =  SparkDataCheck.from_pandas_df(spark, air_data_pd)

In [27]:
# call the count method
air_pd.levels_count("Time_of_Day", "Temp_Range")

,Time_of_Day,Temp_Range,count
0,Mid,Warm,1493
1,Late,Cool,1018
2,Mid,Cool,976
3,Early,Cool,1527
4,Late,Warm,1560
5,Early,Warm,1465
6,Late,NaN,139
7,Mid,NaN,110
8,Early,NaN,117
9,Mid,Hot,537


# Part II - NFL data

## soley using `pandas`-on-Spark

### 1. Read in the weekly nfl data and check first 5 rows

In [35]:
import pyspark.pandas as ps

weekly_nfl_data = ps.read_csv("weekly_nfl_data.csv")

weekly_nfl_data.head()

/home/jupyter-szhou27@ncsu.edu/.local/lib/python3.9/site-packages/pyspark/pandas/utils.py:1037: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `read_csv`, the default index is attached which can cause additional overhead.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)
26/03/09 21:05:33 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


,player_id,player_name,player_display_name,position,position_group,headshot_url,recent_team,season,week,season_type,opponent_team,completions,attempts,passing_yards,passing_tds,interceptions,sacks,sack_yards,sack_fumbles,sack_fumbles_lost,passing_air_yards,passing_yards_after_catch,passing_first_downs,passing_epa,passing_2pt_conversions,pacr,dakota,carries,rushing_yards,rushing_tds,rushing_fumbles,rushing_fumbles_lost,rushing_first_downs,rushing_epa,rushing_2pt_conversions,receptions,targets,receiving_yards,receiving_tds,receiving_fumbles,receiving_fumbles_lost,receiving_air_yards,receiving_yards_after_catch,receiving_first_downs,receiving_epa,receiving_2pt_conversions,racr,target_share,air_yards_share,wopr,special_teams_tds,fantasy_points,fantasy_points_ppr
0,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,1,REG,DEN,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,16,60.0,1,0.0,0.0,4.0,6.248771,0,1,1,7.0,0,0.0,0.0,0.0,0.0,0.0,0.292378,0,0.0,0.052632,NaN,NaN,0.0,12.7,13.7
1,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,2,REG,ARI,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,9,33.0,0,0.0,0.0,1.0,-1.434950,0,3,4,18.0,0,0.0,0.0,0.0,0.0,1.0,0.377009,0,0.0,0.117647,NaN,NaN,0.0,5.1,8.1
2,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,MIA,1999,4,REG,BUF,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,3,2.0,0,0.0,0.0,0.0,-1.539952,0,0,1,0.0,0,0.0,0.0,0.0,0.0,0.0,-0.699578,0,NaN,0.023810,NaN,NaN,0.0,0.2,0.2
3,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,7,REG,LA,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,6,27.0,0,0.0,0.0,0.0,0.216051,0,2,2,8.0,0,0.0,0.0,0.0,0.0,0.0,-0.228454,0,0.0,0.050000,NaN,NaN,0.0,3.5,5.5
4,00-0000003,None,Abdul-Karim al-Jabbar,RB,RB,None,CLE,1999,8,REG,NO,0,0,0.0,0,0.0,0.0,0.0,0,0,0.0,0.0,0.0,NaN,0,NaN,NaN,13,39.0,0,0.0,0.0,2.0,-2.972259,0,0,0,0.0,0,0.0,0.0,0.0,0.0,0.0,NaN,0,NaN,NaN,NaN,NaN,0.0,3.9,3.9


### 2. Report all of the column names

In [37]:
weekly_nfl_data.columns

Index(['player_id', 'player_name', 'player_display_name', 'position',
       'position_group', 'headshot_url', 'recent_team', 'season', 'week',
       'season_type', 'opponent_team', 'completions', 'attempts',
       'passing_yards', 'passing_tds', 'interceptions', 'sacks', 'sack_yards',
       'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards',
       'passing_yards_after_catch', 'passing_first_downs', 'passing_epa',
       'passing_2pt_conversions', 'pacr', 'dakota', 'carries', 'rushing_yards',
       'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost',
       'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions',
       'receptions', 'targets', 'receiving_yards', 'receiving_tds',
       'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards',
       'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa',
       'receiving_2pt_conversions', 'racr', 'target_share', 'air_yards_share',
       'wopr', 'special_teams_tds', 'fantasy_points

### 3. Subst Data

In [53]:
subset_nfl = weekly_nfl_data.loc[
    (weekly_nfl_data["position"] == "QB") & (weekly_nfl_data["season_type"] == "REG") & (weekly_nfl_data["season"].between(2005, 2023)), # conditions
    ["player_display_name", "season", "week", "completions", "attempts", "passing_yards", "passing_tds", "interceptions"] # columns selected
]
subset_nfl.head()

,player_display_name,season,week,completions,attempts,passing_yards,passing_tds,interceptions
29406,Tony Banks,2005,17,14,25,173.0,1,2.0
29426,Charlie Batch,2005,9,9,16,65.0,0,1.0
29427,Charlie Batch,2005,10,13,19,150.0,0,0.0
29428,Charlie Batch,2005,16,1,1,31.0,1,0.0
29447,Jeff Blake,2005,2,1,1,11.0,0,0.0


### 4. Calculate sum and mean

For each `player_display_name` and `season` combination, fine the *sum* and *mean* of each of the statistical quantities (the rest of the columns we chose above)

In [69]:
sum_stats = subset_nfl.groupby(["player_display_name", "season"]).sum()
sum_stats.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jake Delhomme,2006,99,263,431,2805.0,17,11.0
Jake Plummer,2005,144,277,456,3366.0,18,7.0
Matt Schaub,2006,60,18,27,208.0,1,2.0
Vince Young,2006,143,184,356,2199.0,12,13.0
Kerry Collins,2007,48,50,82,531.0,0,0.0


In [70]:
mean_stats = subset_nfl.groupby(["player_display_name", "season"]).mean()
mean_stats.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions
player_display_name,season,,,,,,
Jake Delhomme,2006,7.615385,20.230769,33.153846,215.769231,1.307692,0.846154
Jake Plummer,2005,9.000000,17.312500,28.500000,210.375000,1.125000,0.437500
Matt Schaub,2006,12.000000,3.600000,5.400000,41.600000,0.200000,0.400000
Vince Young,2006,9.533333,12.266667,23.733333,146.600000,0.800000,0.866667
Kerry Collins,2007,8.000000,8.333333,13.666667,88.500000,0.000000,0.000000


### 5. Create two new variables (by season/player combination)

In [71]:
sum_stats["completion_percentage"] = (sum_stats["completions"] / sum_stats["attempts"])

sum_stats["td_int_ratio"] = (sum_stats["passing_tds"] / sum_stats["interceptions"])

sum_stats.head()

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Jake Delhomme,2006,99,263,431,2805.0,17,11.0,0.610209,1.545455
Jake Plummer,2005,144,277,456,3366.0,18,7.0,0.607456,2.571429
Matt Schaub,2006,60,18,27,208.0,1,2.0,0.666667,0.500000
Vince Young,2006,143,184,356,2199.0,12,13.0,0.516854,0.923077
Kerry Collins,2007,48,50,82,531.0,0,0.0,0.609756,NaN


### 6 Another subset

In [73]:
# Subset the rows to only include player/season combinations wher ethe sum of attempts is at least 50.
subset_sum_stats = sum_stats[sum_stats["attempts"] >= 50]

# Sort the rows descending by completion_percentage and report the first 40 values
top_completion = subset_sum_stats.sort_values(
    "completion_percentage",
    ascending = False).head(40)

top_completion

week  completions  attempts  passing_yards  passing_tds  interceptions  completion_percentage  td_int_ratio
player_display_name season                                                                                                             
C.J. Beathard       2023      65           40        53          349.0            1            0.0               0.754717           inf
Colt McCoy          2021      62           74        99          740.0            3            1.0               0.747475      3.000000
Matt Schaub         2019      52           50        67          580.0            3            1.0               0.746269      3.000000
Drew Brees          2018     130          364       489         3992.0           32            5.0               0.744376      6.400000
                    2019     119          281       378         2979.0           27            4.0               0.743386      6.750000
Mason Rudolph       2023      66           55        74          719.0            3            0.0               0.743243           inf
Taysom Hill         2020     147           88       121          928.0            4            2.0               0.727273      2.000000
Nick Foles          2018      51          141       195         1413.0            7            4.0               0.723077      1.750000
Drew Brees          2017     148          386       536         4334.0           23            8.0               0.720149      2.875000
Sam Bradford        2016     146          395       552         3877.0           20            5.0               0.715580      4.000000
Drew Brees          2011     142          471       660         5535.0           46           14.0               0.713636      3.285714
Colt McCoy          2014      57           91       128         1057.0            4            3.0               0.710938      1.333333
Aaron Rodgers       2020     148          372       526         4299.0           48            5.0               0.707224      9.600000
Bailey Zappe        2022      22           65        92          781.0            5            3.0               0.706522      1.666667
Drew Brees          2009     131          363       514         4388.0           34           11.0               0.706226      3.090909
                    2020      97          275       390         2942.0           24            6.0               0.705128      4.000000
Joe Burrow          2021     143          366       520         4611.0           34           14.0               0.703846      2.428571
Derek Carr          2019     147          361       513         4054.0           21            8.0               0.703704      2.625000
Jake Browning       2023     117          171       243         1936.0           12            7.0               0.703704      1.714286
Chase Daniel        2019      20           45        64          435.0            3            2.0               0.703125      1.500000
Ryan Tannehill      2019     128          201       286         2742.0           22            6.0               0.702797      3.666667
Deshaun Watson      2020     145          382       544         4823.0           33            7.0               0.702206      4.714286
Alex Smith          2012      63          153       218         1737.0           13            5.0               0.701835      2.600000
Kirk Cousins        2018     143          425       606         4298.0           30           10.0               0.701320      3.000000
Jamie Martin        2005      92          124       177         1277.0            5            7.0               0.700565      0.714286
Drew Brees          2016     148          471       673         5208.0           37           15.0               0.699851      2.466667
Tony Romo           2014     133          304       435         3705.0           34            9.0               0.698851      3.777778
Matt Ryan           2016     142          373       534         4944.0           38 

In [74]:
# Sort the rows descending by td_int_ratio and report the first 40 values
top_td_int_ratio = subset_sum_stats.sort_values(
    "td_int_ratio",
    ascending = False).head(40)

top_td_int_ratio

,,week,completions,attempts,passing_yards,passing_tds,interceptions,completion_percentage,td_int_ratio
player_display_name,season,,,,,,,,
Charlie Batch,2006,71,30,52,477.0,5,0.0,0.576923,inf
Matt Schaub,2005,65,33,64,495.0,4,0.0,0.515625,inf
Todd Collins,2007,62,67,105,888.0,5,0.0,0.638095,inf
Troy Smith,2007,62,40,76,452.0,2,0.0,0.526316,inf
Jake Locker,2011,51,34,66,542.0,4,0.0,0.515152,inf
Brian Hoyer,2016,27,134,200,1445.0,6,0.0,0.670000,inf
Nick Foles,2016,17,36,55,410.0,3,0.0,0.654545,inf
Derek Anderson,2014,30,65,97,701.0,5,0.0,0.670103,inf
Jimmy Garoppolo,2016,49,43,63,502.0,4,0.0,0.682540,inf
